# NOM / IR / PE — 3×4 Experiment
3 agents, 4 items

In [ ]:
# ── Colab セットアップ（最初に1回だけ実行）──────────────────────────
!git clone https://github.com/shiro-seminar/NOM-matching.git

import sys
sys.path.insert(0, "/content/NOM-matching")

In [ ]:
# コードを更新したいときはここを実行
!git -C /content/NOM-matching pull

In [ ]:
import torch
from nom_ir_pe_3x4.config      import Config
from nom_ir_pe_3x4.train       import train
from nom_ir_pe_3x4.evaluate    import evaluate_mechanism, print_table
from nom_ir_pe_3x4.benchmarks  import BENCHMARKS
from nom_ir_pe_3x4.data_gen    import sample_batch
from nom_ir_pe_3x4.model       import AllocationNet
from nom_ir_pe_3x4.allocations import ir_pe_mask

## 1. Config

In [ ]:
cfg = Config(
    steps      = 50_000,   # 短くしたいなら 5_000 など
    batch_size = 64,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    seed       = 42,
)
print(cfg)
print(f"device: {cfg.device}")

## 2. Train

In [ ]:
net = train(cfg)

## 3. Evaluate — benchmarks vs learned net

In [ ]:
eval_cfg = Config(batch_size=500, device=cfg.device)
torch.manual_seed(0)

batch = sample_batch(eval_cfg)
v, endow_idx, U = batch["v"], batch["endow_idx"], batch["U"]
wmax_w = U.sum(1).max(1).values

results = []
for bname, bfn in BENCHMARKS.items():
    print(f"Evaluating {bname}...")
    results.append(evaluate_mechanism(bname, bfn, eval_cfg, v, endow_idx, U, wmax_w))

net.eval()
def net_mech(cfg_, v_, ei_, U_):
    mask = ir_pe_mask(cfg_, U_, ei_)
    return net(v_, mask=mask, temperature=1e-3)

results.append(evaluate_mechanism("LearnedNet", net_mech, eval_cfg, v, endow_idx, U, wmax_w, is_nn=True))

print_table(results)

## 4. (Optional) チェックポイントを保存 / 読み込み

In [ ]:
# Google Drive にモデルを保存したい場合
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy("alloc_net_3x4.pt", "/content/drive/MyDrive/alloc_net_3x4.pt")
print("saved to Drive")

In [ ]:
# Drive から読み込む場合
CKPT = "/content/drive/MyDrive/alloc_net_3x4.pt"
ckpt = torch.load(CKPT, map_location=cfg.device)
net2 = AllocationNet(Config(**ckpt["cfg"]))
net2.load_state_dict(ckpt["state_dict"])
net2.eval()
print(f"loaded: step={ckpt.get('step', 'final')}")